In [1]:
from youtube_transcript_api import YouTubeTranscriptApi

In [3]:
def get_transcript(video_id):
    transcript = YouTubeTranscriptApi.get_transcript(video_id)
    text = ""
    
    for t in transcript:
        text += t["text"] + " "
        
    return text

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

def create_vector_store(text):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    docs = splitter.split_text(text)

    embeddings = HuggingFaceEmbeddings()

    vectorstore = FAISS.from_texts(docs, embeddings)

    return vectorstore

In [10]:
from langchain_community.llms import HuggingFaceHub

def get_answer(vectorstore, query):

    # retrieve relevant documents
    docs = vectorstore.similarity_search(query, k=3)

    # load LLM
    llm = HuggingFaceHub(
        repo_id="google/flan-t5-base",
        model_kwargs={"temperature": 0.5}
    )

    # combine retrieved text
    context = " ".join([doc.page_content for doc in docs])

    # create prompt
    prompt = f"""
    Use the context below to answer the question.

    Context:
    {context}

    Question:
    {query}
    """

    # generate answer
    response = llm.invoke(prompt)

    return response